In [1]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import random
import os

# ==================================================================================
# 1. CONFIGURATION
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"

BATCH_SIZE = 16  
EPOCHS = 30 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = torch.amp.GradScaler('cuda') 

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 9        
NUM_OBJECTS = 6       

# ==================================================================================
# 2. DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
    def __len__(self): 
        with h5py.File(self.h5_path, 'r') as f: return f['eeg'].shape[0]
    def __getitem__(self, idx):
        with h5py.File(self.h5_path, 'r') as f:
            eeg = torch.from_numpy(f['eeg'][idx].astype(np.float32))
            meta = torch.from_numpy(f['metadata'][idx].astype(np.float32))
            text = torch.from_numpy(f['input_ids'][idx].astype(np.int64))
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg); meta_list.append(meta); text_list.append(txt)
    return torch.stack(eeg_list), torch.stack(meta_list), pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

def create_stratified_split(total_samples, group_size=5):
    # This split is still technically risky for subject leakage, 
    # but we are fixing the architecture bugs first.
    num_groups = total_samples // group_size
    train_indices, val_indices, test_indices = [], [], []
    for group_idx in range(num_groups):
        start_idx = group_idx * group_size
        indices = list(range(start_idx, start_idx + group_size))
        train_indices.extend(indices[:3])
        val_indices.append(indices[3])
        test_indices.append(indices[4])
    return train_indices, val_indices, test_indices

# ==================================================================================
# 3. FAST VECTORIZED MODEL (FIXED)
# ==================================================================================

class DenseGCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj):
        # x: [Batch, Time, Nodes, In_Features]
        x = self.linear(x) 
        # adj: [Nodes, Nodes] - Now using the LEARNED matrix
        return torch.einsum('nm, btni -> btmi', adj, x)

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        
        # --- FIX 1: LEARNABLE GRAPH TOPOLOGY ---
        # Instead of taking a fixed matrix, we learn the weights between electrodes.
        # We initialize randomly to break symmetry so it can learn specific paths.
        self.adaptive_adj = nn.Parameter(torch.randn(num_channels, num_channels), requires_grad=True)
        
        self.gcn1 = DenseGCNLayer(1, enc_hidden) 
        self.gcn2 = DenseGCNLayer(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg):
        # 1. Normalize the learned adjacency matrix to make it stable (Graph Softmax)
        # This ensures the weights sum to 1 for each node (like a probability distribution)
        adj = F.softmax(self.adaptive_adj, dim=1)
        
        # eeg: [Batch, Channels, Time] -> [Batch, Time, Channels, 1]
        x = eeg.permute(0, 2, 1).unsqueeze(-1) 
        
        # Pass the LEARNED 'adj' into the GCN layers
        x = F.relu(self.gcn1(x, adj))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, adj)) 
        
        # Mean Pool over Channels: [Batch, Time, Hidden]
        x = torch.mean(x, dim=2) 
        
        encoder_outputs, encoder_hidden = self.rnn(x)
        return encoder_outputs.permute(1, 0, 2), encoder_hidden

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects):
        super().__init__()
        self.color_p = nn.Sequential(nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, 32))
        self.obj_p = nn.Sequential(nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 32))
        
    def forward(self, m): 
        # This can now accept Hard Labels (0/1) OR Soft Probabilities (0.0 - 1.0)
        return torch.cat([self.color_p(m[:,:9]), self.obj_p(m[:,9:])], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = nn.Linear(enc_hidden * 2, dec_hidden)
        self.rnn = nn.GRU(emb_dim + enc_hidden * 2 + meta_dim + enc_hidden * 2, dec_hidden, num_layers, dropout=dropout)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_hidden * 2, dec_hidden)
        self.num_layers = num_layers
    
    def init_hidden(self, enc_hid):
        hidden = enc_hid.view(self.num_layers, 2, enc_hid.size(1), -1)
        return torch.tanh(self.bridge(torch.cat((hidden[-1][0], hidden[-1][1]), dim=1))).unsqueeze(0).repeat(self.num_layers, 1, 1)
    
    def forward(self, tok, dec_hid, enc_out, meta, glob_ctx):
        embedded = self.dropout(self.embedding(tok.unsqueeze(0)))
        scores = torch.bmm(dec_hid[-1].unsqueeze(0).permute(1, 0, 2), self.attention(enc_out).permute(1, 2, 0))
        context = torch.bmm(F.softmax(scores, dim=2), enc_out.permute(1, 0, 2)).permute(1, 0, 2)
        return self.fc_out(self.rnn(torch.cat((embedded, context, meta.unsqueeze(0), glob_ctx.unsqueeze(0)), dim=2), dec_hid)[0].squeeze(0)), self.rnn(torch.cat((embedded, context, meta.unsqueeze(0), glob_ctx.unsqueeze(0)), dim=2), dec_hid)[1], context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, num_colors, num_objects, pad_id):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(num_channels=62, enc_hidden=256) 
        self.meta_encoder = MetadataEncoder(num_colors, num_objects)
        self.decoder = Decoder(vocab_size, 256, 256, 256, 64, 2, pad_id, 0.2)
        self.meta_head = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.3), nn.Linear(256, num_colors + num_objects))
        self.num_colors = num_colors
        self.vocab_size = vocab_size

    def forward(self, eeg, metadata, target_text, tf_ratio=0.5, inference_mode=False):
        # 1. Encode (Learns topology internally now)
        enc_out, enc_hid = self.encoder(eeg)
        
        # 2. Global Context for Metadata Prediction
        glob_ctx = torch.cat((enc_hid.view(2, 2, eeg.size(0), -1)[-1][0], enc_hid.view(2, 2, eeg.size(0), -1)[-1][1]), dim=1)
        meta_preds = self.meta_head(glob_ctx)
        
        if inference_mode:
            pred_probs = torch.sigmoid(meta_preds) 
            meta_feat = self.meta_encoder(pred_probs)
        else:
            meta_feat = self.meta_encoder(metadata)

        dec_hid = self.decoder.init_hidden(enc_hid)
        outputs = torch.zeros(target_text.shape[1], eeg.size(0), self.vocab_size).to(eeg.device)
        dec_input = target_text[:, 0]
        
        for t in range(1, target_text.shape[1]):
            out, dec_hid, _ = self.decoder(dec_input, dec_hid, enc_out, meta_feat, glob_ctx)
            outputs[t] = out
            dec_input = target_text[:, t] if random.random() < tf_ratio else out.argmax(1)
            
        return outputs[1:].permute(1, 0, 2), meta_preds[:, :self.num_colors], meta_preds[:, self.num_colors:]

# ==================================================================================
# 4. TRAINING
# ==================================================================================
if __name__ == "__main__":
    torch.cuda.empty_cache()
    
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    train_idx, val_idx, test_idx = create_stratified_split(len(dataset))
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch, num_workers=2, pin_memory=True)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch, num_workers=2, pin_memory=True)
    
    # NOTE: We no longer create adj_matrix here. The model creates it internally.

    model = Seq2Seq(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS, PAD_ID).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    criterion_t = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    criterion_m = nn.BCEWithLogitsLoss()
    
    print("\n--- Starting Adaptive Graph Training (No Leakage) ---")
    best_val_loss = float('inf')
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        optimizer.zero_grad()
        
        for eeg, meta, txt in tqdm(train_loader, leave=False):
            eeg, meta, txt = eeg.to(device), meta.to(device), txt.to(device)
            
            with torch.amp.autocast('cuda'):
                txt_logits, c_pred, o_pred = model(eeg, meta, txt, tf_ratio=max(0.2, 1.0-epoch/10), inference_mode=False)
                
                loss = criterion_t(txt_logits.reshape(-1, TEXT_VOCAB_SIZE), txt[:,1:].reshape(-1)) + \
                       criterion_m(c_pred, meta[:,:9]) + criterion_m(o_pred, meta[:,9:])
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            total_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for eeg, meta, txt in val_loader:
                eeg, meta, txt = eeg.to(device), meta.to(device), txt.to(device)
                with torch.amp.autocast('cuda'):
                    txt_logits, c_pred, o_pred = model(eeg, meta, txt, tf_ratio=0, inference_mode=True)
                    
                    loss = criterion_t(txt_logits.reshape(-1, TEXT_VOCAB_SIZE), txt[:,1:].reshape(-1)) + \
                           criterion_m(c_pred, meta[:,:9]) + criterion_m(o_pred, meta[:,9:])
                val_loss += loss.item()
        
        avg_val_loss = val_loss/len(val_loader)
        print(f"Epoch {epoch} | Loss: {total_loss/len(train_loader):.3f} | Val: {avg_val_loss:.3f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'adaptive-graph-phase-1.pt')


--- Starting Adaptive Graph Training (No Leakage) ---


  0%|          | 0/1050 [00:00<?, ?it/s]

Epoch 1 | Loss: 6.421 | Val: 5.562


  0%|          | 0/1050 [00:00<?, ?it/s]

Epoch 2 | Loss: 5.229 | Val: 5.373


...

Epoch 9 | Loss: 4.382 | Val: 5.224


  0%|          | 0/1050 [00:00<?, ?it/s]

Epoch 10 | Loss: 4.305 | Val: 5.315


  0%|          | 0/1050 [00:00<?, ?it/s]

KeyboardInterrupt: 